# Capstone 4 — Manufacturing IoT Predictive Maintenance
### Microsoft Fabric Medallion + Structured Streaming Capstone

**Business scenario:** "IronWorks Manufacturing" has sensors on its factory floor machines streaming temperature and vibration readings. They want early warning of machines heading toward failure.

**Fabric capability highlighted:** **Spark Structured Streaming** reading incrementally-arriving sensor files into Bronze (the same pattern used for Eventstream/Kafka sources), feeding a Medallion pipeline with an anomaly-detection Gold mart.

**What this notebook builds:**
1. Synthetic machines + a stream of sensor readings written in small batches (simulating live telemetry)
2. Bronze — ingested via `readStream`/`writeStream` (structured streaming), not a one-shot batch read
3. Silver — cleansed readings with out-of-range values quarantined
4. Gold — `mart_anomaly_flags` (statistical outlier detection per machine) and `mart_maintenance_summary`

Attach this notebook to a Lakehouse (e.g. `manufacturing_capstone_lakehouse`) before running.

In [ ]:
import random
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql import functions as F

random.seed(33)
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

NUM_MACHINES = 25
STREAM_PATH = "Files/streaming/sensor_readings"
CHECKPOINT_PATH = "Files/streaming/_checkpoints/sensor_readings"
BATCHES_TO_SIMULATE = 12
READINGS_PER_MACHINE_PER_BATCH = 5

## 1. Machine master data

In [ ]:
machine_types = ["CNC Lathe", "Hydraulic Press", "Conveyor Motor", "Industrial Robot Arm", "Welding Station"]

machines = [{
    "machine_id": f"MCH{i:03d}",
    "machine_type": random.choice(machine_types),
    "install_date": (datetime.utcnow() - timedelta(days=random.randint(200, 3000))).date().isoformat(),
    "line": random.choice(["Line A", "Line B", "Line C"]),
    # a handful of machines are pre-designated as "degrading" so the anomaly mart has something to catch
    "_degrading": random.random() < 0.15,
} for i in range(1, NUM_MACHINES + 1)]

df_machines = pd.DataFrame(machines)
spark.createDataFrame(df_machines.drop(columns=["_degrading"])).write.format("delta") \
    .mode("overwrite").saveAsTable("bronze.machines")
print(f"{len(df_machines)} machines registered ({df_machines['_degrading'].sum()} simulated as degrading)")

## 2. Simulate a sensor telemetry stream
Writes one small JSON batch of readings per iteration to `Files/streaming/sensor_readings/`, mimicking what a real Eventstream-to-Files landing or an IoT Hub capture would produce.

In [ ]:
def generate_batch(batch_num):
    rows = []
    ts = datetime.utcnow()
    for _, m in df_machines.iterrows():
        for _ in range(READINGS_PER_MACHINE_PER_BATCH):
            # Degrading machines drift toward higher temperature & vibration over successive batches
            drift = (batch_num / BATCHES_TO_SIMULATE) * 25 if m["_degrading"] else 0
            rows.append({
                "machine_id": m["machine_id"],
                "reading_timestamp": (ts + timedelta(seconds=random.randint(0, 59))).isoformat(),
                "temperature_c": round(random.gauss(65 + drift, 4), 2),
                "vibration_mm_s": round(max(0, random.gauss(2.5 + drift / 8, 0.5)), 2),
                "rpm": random.randint(800, 3000),
            })
    pdf = pd.DataFrame(rows)
    spark.createDataFrame(pdf).coalesce(1).write.mode("append").json(f"{STREAM_PATH}/batch_{batch_num:03d}")
    return len(rows)

for b in range(BATCHES_TO_SIMULATE):
    n = generate_batch(b)
print(f"Simulated {BATCHES_TO_SIMULATE} telemetry batches ({n} readings each) under {STREAM_PATH}/")

## 3. Bronze layer — ingest via Structured Streaming
`readStream` picks up every JSON batch written above as if it were arriving live; `trigger(availableNow=True)` processes everything currently sitting in the folder and then stops (the standard pattern for a scheduled micro-batch pipeline).

In [ ]:
schema = spark.read.json(f"{STREAM_PATH}/batch_000").schema

stream_df = (spark.readStream
    .schema(schema)
    .json(STREAM_PATH))

query = (stream_df
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("bronze.sensor_readings"))

query.awaitTermination()
print(f"bronze.sensor_readings: {spark.table('bronze.sensor_readings').count():,} rows ingested via streaming")

## 4. Silver layer — quarantine physically implausible readings

In [ ]:
bronze_readings = (spark.table("bronze.sensor_readings")
    .withColumn("reading_timestamp", F.to_timestamp("reading_timestamp")))

# Anything outside a plausible operating envelope is quarantined, not silently kept
plausible = (F.col("temperature_c").between(-20, 150)) & (F.col("vibration_mm_s").between(0, 50))

passed = bronze_readings.filter(plausible)
failed = bronze_readings.filter(~plausible).withColumn("_dq_reason", F.lit("reading outside plausible sensor range"))

failed.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable("silver.sensor_readings_quarantine")
passed.write.format("delta").mode("overwrite").saveAsTable("silver.sensor_readings")

print(f"silver.sensor_readings: {passed.count():,} passed | {failed.count():,} quarantined")

## 5. Gold layer — anomaly detection & maintenance summary

In [ ]:
from pyspark.sql.window import Window

readings = spark.table("silver.sensor_readings")

stats = (readings.groupBy("machine_id")
    .agg(F.avg("temperature_c").alias("avg_temp"), F.stddev("temperature_c").alias("std_temp"),
         F.avg("vibration_mm_s").alias("avg_vibe"), F.stddev("vibration_mm_s").alias("std_vibe")))

# Flag readings more than 2 standard deviations above the machine's own mean (simple z-score anomaly rule)
mart_anomaly_flags = (readings
    .join(stats, "machine_id")
    .withColumn("temp_zscore", (F.col("temperature_c") - F.col("avg_temp")) / F.col("std_temp"))
    .withColumn("vibe_zscore", (F.col("vibration_mm_s") - F.col("avg_vibe")) / F.col("std_vibe"))
    .withColumn("is_anomalous", (F.col("temp_zscore") > 2) | (F.col("vibe_zscore") > 2))
    .filter("is_anomalous = true")
    .join(spark.table("bronze.machines"), "machine_id")
    .select("machine_id", "machine_type", "line", "reading_timestamp",
            "temperature_c", "vibration_mm_s", "temp_zscore", "vibe_zscore"))

mart_anomaly_flags.write.format("delta").mode("overwrite").saveAsTable("gold.mart_anomaly_flags")

mart_maintenance_summary = (mart_anomaly_flags
    .groupBy("machine_id", "machine_type", "line")
    .agg(F.count("*").alias("anomalous_reading_count"),
         F.max("reading_timestamp").alias("last_anomaly_at"))
    .withColumn("maintenance_priority",
                F.when(F.col("anomalous_reading_count") >= 20, "Urgent")
                 .when(F.col("anomalous_reading_count") >= 5, "Schedule Soon")
                 .otherwise("Monitor"))
    .orderBy(F.desc("anomalous_reading_count")))

mart_maintenance_summary.write.format("delta").mode("overwrite").saveAsTable("gold.mart_maintenance_summary")

print(f"gold.mart_anomaly_flags: {mart_anomaly_flags.count():,} anomalous readings")
display(mart_maintenance_summary)

## 6. Capstone checkpoint
Suggested Power BI pages: **Fleet Health** (anomaly count by line/machine type), **Maintenance Priority Queue** over `mart_maintenance_summary`, and a **Sensor Trend** line chart per machine for engineers to drill into.

In [ ]:
for t in ["bronze.machines", "bronze.sensor_readings", "silver.sensor_readings",
          "gold.mart_anomaly_flags", "gold.mart_maintenance_summary"]:
    print(f"{t:35s} -> {spark.table(t).count():,} rows")
print("\nCapstone 4 (Manufacturing IoT Predictive Maintenance) complete.")